In [24]:
from functools import reduce
import pandas as pd
import os
import numpy as np

In [25]:
files = {
    "usd_php":    "../data/semi-cleaned/fx_usd_php.parquet",
    "rub_usd":    "../data/semi-cleaned/fx_rub_usd.parquet",
    "brent":      "../data/semi-cleaned/brent_weekly.parquet",
    "wti":        "../data/semi-cleaned/wti_weekly.parquet",
    "dubai":      "../data/semi-cleaned/dubai_crude_weekly.parquet",
    "mops_gasoil":"../data/semi-cleaned/mops_gasoil_weekly.parquet",
    "mops_rbob":  "../data/semi-cleaned/mops_rbob_weekly.parquet",
    "cpi":        "../data/semi-cleaned/full_cpi.parquet",
}

In [26]:
for feature, path in files.items():
    df = pd.read_parquet(path)
    print(f"{feature}")
    print(f"index name: {df.index.name}")
    print(f"index dtype: {df.index.dtype}")
    print(f"date range: {df.index.min()} -> {df.index.max()}")
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
    print(f"nulls: {df.isnull().sum().to_dict()}")
    print()


usd_php
index name: week_monday
index dtype: datetime64[ns]
date range: 2015-01-05 00:00:00 -> 2026-05-11 00:00:00
shape: (593, 2)
columns: ['usd_php_close', 'usd_php_mean']
nulls: {'usd_php_close': 0, 'usd_php_mean': 0}

rub_usd
index name: week_monday
index dtype: datetime64[ns]
date range: 2015-01-05 00:00:00 -> 2026-05-11 00:00:00
shape: (593, 2)
columns: ['rub_usd_close', 'rub_usd_mean']
nulls: {'rub_usd_close': 0, 'rub_usd_mean': 0}

brent
index name: week_monday
index dtype: datetime64[ns]
date range: 2017-01-09 00:00:00 -> 2026-05-04 00:00:00
shape: (487, 2)
columns: ['brent_close', 'brent_mean']
nulls: {'brent_close': 0, 'brent_mean': 0}

wti
index name: week_monday
index dtype: datetime64[ns]
date range: 2017-01-09 00:00:00 -> 2026-05-04 00:00:00
shape: (487, 2)
columns: ['wti_close', 'wti_mean']
nulls: {'wti_close': 0, 'wti_mean': 0}

dubai
index name: week_monday
index dtype: datetime64[ns]
date range: 2017-01-09 00:00:00 -> 2026-05-11 00:00:00
shape: (488, 1)
columns: ['du

In [27]:
# join everything
dfs = [
    pd.read_parquet("../data/semi-cleaned/fx_usd_php.parquet"),
    pd.read_parquet("../data/semi-cleaned/brent_weekly.parquet"),
    pd.read_parquet("../data/semi-cleaned/wti_weekly.parquet"),
    pd.read_parquet("../data/semi-cleaned/mops_gasoil_weekly.parquet"),
    pd.read_parquet("../data/semi-cleaned/mops_rbob_weekly.parquet"),
    pd.read_parquet("../data/semi-cleaned/dubai_crude_weekly.parquet"),
    pd.read_parquet("../data/semi-cleaned/full_cpi.parquet"),
    pd.read_parquet("../data/semi-cleaned/fx_rub_usd.parquet"),
]

features = reduce(lambda l, r: l.join(r, how="outer"), dfs)
features = features.loc["2018-01-01":]

In [28]:
features = features.drop(columns=["usd_php_close", "brent_close", "wti_close", "mops_gasoil_close", "mops_rbob_close", "rub_usd_close"])

In [29]:
features 

,usd_php_mean,brent_mean,wti_mean,mops_gasoil_mean,mops_rbob_mean,dubai_close,cpi_value,rub_usd_mean
week_monday,,,,,,,,
2018-01-01,49.837600,66.520,59.8800,75.340498,75.288150,61.384286,96.348116,0.017285
2018-01-08,49.859200,67.810,61.3625,80.141001,75.106500,66.024783,97.200000,0.017434
2018-01-15,50.112200,69.470,63.2560,80.935800,76.836480,66.024783,97.200000,0.017577
2018-01-22,50.474200,69.388,63.7700,81.468500,78.160950,66.024783,97.200000,0.017689
2018-01-29,50.813000,70.040,65.1380,81.789400,80.289720,66.024783,97.200000,0.017765
...,...,...,...,...,...,...,...,...
2026-04-13,59.842799,124.605,104.5440,209.639999,131.521320,126.706364,127.200000,0.012655
2026-04-20,59.952801,114.432,93.8400,191.773001,129.302881,126.706364,127.200000,0.013160
2026-04-27,60.025600,109.618,95.4300,192.018002,139.517280,126.706364,127.200000,0.013279


In [30]:
features.corr()

,usd_php_mean,brent_mean,wti_mean,mops_gasoil_mean,mops_rbob_mean,dubai_close,cpi_value,rub_usd_mean
usd_php_mean,1.000000,0.469456,0.455727,0.561476,0.440151,0.537959,0.811900,-0.303285
brent_mean,0.469456,1.000000,0.988717,0.938220,0.950275,0.965101,0.409976,-0.081220
wti_mean,0.455727,0.988717,1.000000,0.910126,0.960503,0.951823,0.443681,-0.117249
mops_gasoil_mean,0.561476,0.938220,0.910126,1.000000,0.894931,0.947217,0.445217,-0.033585
mops_rbob_mean,0.440151,0.950275,0.960503,0.894931,1.000000,0.925007,0.448575,-0.079895
dubai_close,0.537959,0.965101,0.951823,0.947217,0.925007,1.000000,0.473361,-0.129476
cpi_value,0.811900,0.409976,0.443681,0.445217,0.448575,0.473361,1.000000,-0.525184
rub_usd_mean,-0.303285,-0.081220,-0.117249,-0.033585,-0.079895,-0.129476,-0.525184,1.000000


### DOE Cleaning and Aggregation

In [31]:
df = pd.read_parquet("../data/semi-cleaned/processed_doe_data.parquet")

In [32]:
df

,Monitoring Dates,Effectivity Date,City,Product,Brand,Price Low (P/L),Price High (P/L),Notes
0,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Petron,52.40,52.40,
1,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Shell,56.85,56.85,
2,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Caltex,47.40,47.40,
3,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Phoenix,54.01,54.83,
4,"April 27-28, 2021","Apr 27, 2021",Caloocan City,RON 91,Total,56.80,56.80,
...,...,...,...,...,...,...,...,...
125232,"September 15-16, 2020","Sep 15, 2020",Muntinlupa City,Diesel Plus,Independent,34.34,34.34,
125233,"September 15-16, 2020","Sep 15, 2020",Muntinlupa City,Kerosene,Petron,34.48,34.48,
125234,"September 15-16, 2020","Sep 15, 2020",Muntinlupa City,Kerosene,Shell,34.20,34.20,
125235,"September 15-16, 2020","Sep 15, 2020",Muntinlupa City,Kerosene,Caltex,34.20,34.20,


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125237 entries, 0 to 125236
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Monitoring Dates  125237 non-null  object 
 1   Effectivity Date  125237 non-null  object 
 2   City              125237 non-null  object 
 3   Product           125237 non-null  object 
 4   Brand             125237 non-null  object 
 5   Price Low (P/L)   125237 non-null  float64
 6   Price High (P/L)  125237 non-null  float64
 7   Notes             125237 non-null  object 
dtypes: float64(2), object(6)
memory usage: 7.6+ MB


In [34]:
# Fix dates 
df["effectivity_date"] = pd.to_datetime(df["Effectivity Date"], errors="coerce")

typo_map = {
    "March 24,2020":       "2020-03-24",
    "February 18,2020":    "2020-02-18",
    "Decembeer 18, 2018":  "2018-12-18",
    "Decembeer 25, 2018":  "2018-12-25",
    "January 001, 2019":   "2019-01-01",
    "September 111, 2018": "2018-09-11",
}
mask = df["effectivity_date"].isna()
df.loc[mask, "effectivity_date"] = df.loc[mask, "Effectivity Date"].map(typo_map)
df["effectivity_date"] = pd.to_datetime(df["effectivity_date"], errors="coerce")
df = df.dropna(subset=["effectivity_date"])

In [35]:
# NCR cities only
ncr_cities = {
    "Taguig City", "Manila City", "MANILA City",
    "Caloocan City", "Paranaque City", "Pasig City",
    "Quezon City", "Quezon  City",
    "Muntinlupa City", "Pasay City", "Makati City",
    "Marikina City", "Navotas City", "Valenzuela City",
    "Metro Manila",
}
df = df[df["City"].isin(ncr_cities)].copy()

In [36]:
# Ratio-based corruption filter ─────────────────────────────────────────
# Price High / Price Low ratio distribution has a natural break at 1.5:
# - p99.9 across ALL periods (including COVID, Russia-Ukraine, 2026 tariff) ≤ 1.55
# - 71 corrupt rows all cluster above 2.0, max ratio = 12.6
# - Threshold of 1.5 sits cleanly in that gap — data-driven, not arbitrary
# For corrupt rows: Price High is the bad column, Price Low is clean throughout
# so we fall back to Price Low as both bounds for those 71 rows
df["price_ratio"] = df["Price High (P/L)"] / df["Price Low (P/L)"]

df_1 = df.copy()
corrupt_mask = df["price_ratio"] > 1.5
print(f"Corrupt Price High rows flagged: {corrupt_mask.sum()}")  # expect 71

df["price_high_clean"] = df["Price High (P/L)"].where(~corrupt_mask,
                                                        other=df["Price Low (P/L)"])
df["price_mid"] = (df["Price Low (P/L)"] + df["price_high_clean"]) / 2

# Hard absolute bounds — belt and suspenders
df = df[(df["price_mid"] >= 20) & (df["price_mid"] <= 200)].copy()

Corrupt Price High rows flagged: 100


In [37]:
check = df_1[corrupt_mask]
check

,Monitoring Dates,Effectivity Date,City,Product,Brand,Price Low (P/L),Price High (P/L),Notes,effectivity_date,price_ratio
2493,"February 3-6, 2026","Feb 03, 2026",Pasay City,Ron95,Total,50.00,75.89,,2026-02-03,1.517800
2895,"February 10-13, 2026","Feb 10, 2026",Pasay City,Ron95,Total,50.00,76.49,,2026-02-10,1.529800
3296,"February 17-20, 2026","Feb 17, 2026",Pasay City,Ron95,Total,50.00,77.64,,2026-02-17,1.552800
7982,"April 22-25,2025","Apr 22, 2025",Pasay City,Ron95,Total,45.00,71.50,,2025-04-22,1.588889
8730,"April 29 - May 2,2025","Apr 29, 2025",Pasay City,Ron95,Total,45.00,72.85,,2025-04-29,1.618889
...,...,...,...,...,...,...,...,...,...,...
24394,,"Jan 15, 2019",Metro Manila,Kerosene,Seaoil,40.62,500.90,,2019-01-15,12.331364
34447,"August 18-19, 2020","Aug 18, 2020",Muntinlupa City,RON 91,Unioil,34.41,52.50,,2020-08-18,1.525719
34555,"August 25-26, 2020","Aug 25, 2020",Manila City,RON 91,Caltex,47.95,79.65,,2020-08-25,1.661105
76700,"November 15-17, 2022","Nov 15, 2022",Manila City,Diesel Plus,Petron,75.00,850.31,,2022-11-15,11.337467


In [38]:
# Product standardization
df["product"] = (
    df["Product"]
    .str.upper()
    .str.replace(r"\s+", "", regex=True)
    .str.replace("-", "")
)
product_map = {
    "RON91": "ron91", "RON95": "ron95", "RON97": "ron97",
    "RON100": "ron100", "DIESEL": "diesel",
    "DIESELPLUS": "diesel_plus", "KEROSENE": "kerosene",
}
df["product"] = df["product"].map(product_map)
df = df.dropna(subset=["product"])

In [39]:
# Weekly median per product using midpoint price
df["week_monday"] = df["effectivity_date"] - pd.Timedelta(days=1)

doe_weekly = (
    df.groupby(["week_monday", "product"])["price_mid"]
    .median()
    .unstack("product")
    .sort_index()
)

In [40]:
# Sample window 
doe_weekly = doe_weekly.loc["2018-01-01":]

In [41]:
# Sanity checks
print("Shape:", doe_weekly.shape)
print("\nFirst date:", doe_weekly.index.min())
print("Last date: ", doe_weekly.index.max())
print("\nNull counts:\n", doe_weekly.isnull().sum())
print("\nDiesel median by year:")
print(doe_weekly["diesel"].resample("YE").median().round(2))

Shape: (383, 7)

First date: 2018-02-12 00:00:00
Last date:  2026-04-27 00:00:00

Null counts:
 product
diesel          5
diesel_plus     5
kerosene        5
ron100         35
ron91           0
ron95           0
ron97           0
dtype: int64

Diesel median by year:
week_monday
2018-12-31    42.30
2019-12-31    42.04
2020-12-31    34.97
2021-12-31    43.53
2022-12-31    74.11
2023-12-31    60.02
2024-12-31    58.40
2025-12-31    56.25
2026-12-31    58.68
Freq: YE-DEC, Name: diesel, dtype: float64


In [42]:
doe_weekly 

product,diesel,diesel_plus,kerosene,ron100,ron91,ron95,ron97
week_monday,,,,,,,
2018-02-12,39.175,43.550,48.350,NaN,50.135,52.0700,54.075
2018-02-19,38.025,41.980,45.790,NaN,49.085,50.9950,53.020
2018-02-26,38.925,43.070,48.935,NaN,49.735,51.6700,53.675
2018-03-05,39.175,43.430,49.735,NaN,50.195,52.1700,54.175
2018-03-13,38.950,42.880,49.735,NaN,49.950,51.8200,53.825
...,...,...,...,...,...,...,...
2026-03-30,131.550,140.000,154.750,102.950,93.700,98.3025,107.300
2026-04-06,147.800,155.700,163.050,104.500,94.950,97.1000,106.425
2026-04-13,123.600,129.500,155.300,97.625,87.150,90.3000,99.050


In [43]:
doe_weekly.to_parquet("../data/semi-cleaned/doe_weekly.parquet")

In [44]:
doe = pd.read_parquet("../data/semi-cleaned/doe_weekly.parquet")
# effectivity_date was already dropped and week_monday is already the index
# just join directly
master = features.join(doe, how="inner")


os.makedirs("../data/final", exist_ok=True)
master.to_parquet("../data/final/master_weekly.parquet")
print("Saved. Shape:", master.shape)
master.tail()

Saved. Shape: (373, 15)


,usd_php_mean,brent_mean,wti_mean,mops_gasoil_mean,mops_rbob_mean,dubai_close,cpi_value,rub_usd_mean,diesel,diesel_plus,kerosene,ron100,ron91,ron95,ron97
week_monday,,,,,,,,,,,,,,,
2026-03-30,59.841000,111.242,94.292,189.149402,130.331038,126.706364,127.2,0.012262,131.550,140.000,154.75,102.950,93.700,98.3025,107.300
2026-04-06,60.456999,123.935,105.670,204.551247,136.953600,126.706364,127.2,0.012380,147.800,155.700,163.05,104.500,94.950,97.1000,106.425
2026-04-13,59.842799,124.605,104.544,209.639999,131.521320,126.706364,127.2,0.012655,123.600,129.500,155.30,97.625,87.150,90.3000,99.050
2026-04-20,59.952801,114.432,93.840,191.773001,129.302881,126.706364,127.2,0.013160,98.700,100.525,146.95,92.400,82.990,84.6500,92.400
2026-04-27,60.025600,109.618,95.430,192.018002,139.517280,126.706364,127.2,0.013279,85.275,87.800,131.45,89.225,81.775,82.5250,91.500


In [45]:
master

,usd_php_mean,brent_mean,wti_mean,mops_gasoil_mean,mops_rbob_mean,dubai_close,cpi_value,rub_usd_mean,diesel,diesel_plus,kerosene,ron100,ron91,ron95,ron97
week_monday,,,,,,,,,,,,,,,
2018-02-12,51.383200,65.496,62.014,79.039999,74.617200,62.631000,97.9,0.017446,39.175,43.550,48.350,NaN,50.135,52.0700,54.075
2018-02-19,51.917200,62.718,60.562,76.596400,71.933401,62.631000,97.9,0.017438,38.025,41.980,45.790,NaN,49.085,50.9950,53.020
2018-02-26,52.081001,65.520,62.470,77.809500,74.358901,62.631000,97.9,0.017698,38.925,43.070,48.935,NaN,49.735,51.6700,53.675
2018-03-05,51.844800,66.024,62.070,76.870601,77.159039,63.917273,98.4,0.017777,39.175,43.430,49.735,NaN,50.195,52.1700,54.175
2018-04-23,52.051800,73.618,67.546,83.669800,86.710680,68.667143,98.8,0.016312,42.300,46.325,49.440,NaN,51.870,53.3700,55.795
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-30,59.841000,111.242,94.292,189.149402,130.331038,126.706364,127.2,0.012262,131.550,140.000,154.750,102.950,93.700,98.3025,107.300
2026-04-06,60.456999,123.935,105.670,204.551247,136.953600,126.706364,127.2,0.012380,147.800,155.700,163.050,104.500,94.950,97.1000,106.425
2026-04-13,59.842799,124.605,104.544,209.639999,131.521320,126.706364,127.2,0.012655,123.600,129.500,155.300,97.625,87.150,90.3000,99.050
